# Fine-tuning Gemma 4 E4B for Tool Use (QLoRA)

This notebook fine-tunes `google/gemma-4-e4b-it` on the Salesforce xLAM-60k function-calling dataset using QLoRA — all on a **free Colab T4 GPU**.

**Before running:**
1. Runtime → Change runtime type → **T4 GPU**
2. Fill in your secrets in the cell below (HF_TOKEN, WANDB_API_KEY, GROQ_API_KEY, GEMINI_API_KEY)

**Expected run time:** ~60–90 minutes for 1 epoch on ~30k examples.

**If Colab disconnects:** Just re-run all cells. The training script automatically resumes from the last saved checkpoint.

## Step 1 — Check GPU
First thing: verify Colab gave us an actual GPU. If this shows `No GPU`, go to Runtime → Change runtime type → T4 GPU.

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
if result.returncode == 0:
    print(f'GPU found: {result.stdout.strip()}')
else:
    print('No GPU detected! Go to Runtime → Change runtime type → T4 GPU')

## Step 2 — Install dependencies

Colab comes with Python and PyTorch pre-installed, but we need several additional libraries:
- `transformers` — Hugging Face's model library (loads Gemma)
- `peft` — the LoRA library (attaches adapter layers)
- `trl` — the SFT training library
- `bitsandbytes` — enables 4-bit quantization
- `accelerate` — helps distribute training across hardware
- `datasets` — loads our JSONL files
- `wandb` — experiment tracking dashboard

This cell takes about 2–3 minutes. The `-q` flag suppresses verbose output.

In [ ]:
%%capture install_output
# %%capture suppresses the noisy pip output. Remove it if you want to see install logs.
!pip install -q \
    transformers>=4.44.0 \
    peft>=0.12.0 \
    trl>=0.9.0 \
    bitsandbytes>=0.43.0 \
    accelerate>=0.31.0 \
    datasets>=2.19.0 \
    wandb>=0.17.0 \
    huggingface-hub>=0.23.0 \
    sentencepiece>=0.2.0 \
    protobuf>=5.27.0 \
    python-dotenv>=1.0.0 \
    pyyaml>=6.0

print('Installation complete.')

## Step 3 — Set your secrets

Replace the placeholder strings with your real keys. These are stored only in this session — never committed to git.

Where to get them:
- **HF_TOKEN**: huggingface.co → Settings → Access Tokens → New token (write access)
- **WANDB_API_KEY**: wandb.ai → Settings → API keys
- **GROQ_API_KEY**: console.groq.com → API Keys (free)
- **GEMINI_API_KEY**: aistudio.google.com → Get API key (free)

In [ ]:
import os

os.environ['HF_TOKEN']       = 'hf_YOUR_TOKEN_HERE'
os.environ['WANDB_API_KEY']  = 'YOUR_WANDB_KEY_HERE'
os.environ['GROQ_API_KEY']   = 'YOUR_GROQ_KEY_HERE'
os.environ['GEMINI_API_KEY'] = 'YOUR_GEMINI_KEY_HERE'

# Verify the HF token at least looks right
token = os.environ['HF_TOKEN']
if token.startswith('hf_') and len(token) > 20:
    print('HF_TOKEN looks valid.')
else:
    print('WARNING: HF_TOKEN does not look right. Check it.')

## Step 4 — Clone the repo

We clone your GitHub repo into Colab's temporary filesystem. This gives us `train.py`, `configs/train.yaml`, and all the project code.

If the repo already exists (e.g. you're resuming after a disconnect), we just pull the latest changes instead.

In [ ]:
import os
from pathlib import Path

REPO_URL  = 'https://github.com/sahilmdeshmukh/function-call-finetune.git'
REPO_NAME = 'function-call-finetune'

if Path(REPO_NAME).exists():
    print('Repo already cloned. Pulling latest...')
    !git -C {REPO_NAME} pull
else:
    print('Cloning repo...')
    !git clone {REPO_URL}

# Change into the repo directory for all subsequent cells
os.chdir(REPO_NAME)
print(f'Working directory: {os.getcwd()}')

## Step 5 — Prepare the dataset

This downloads the xLAM-60k dataset from Hugging Face, converts it from Llama format to Gemma format, and saves the train/val/test splits to `data/`.

Takes ~10 minutes (most of that is the download). Creates:
- `data/train.jsonl` — ~49,000 examples for training
- `data/val.jsonl` — ~2,700 examples for validation  
- `data/test_held_out.jsonl` — 500 examples for final eval (never seen during training)

If `data/train.jsonl` already exists (resuming after a disconnect), skip this cell.

In [ ]:
from pathlib import Path

if Path('data/train.jsonl').exists():
    print('Data already prepared. Skipping.')
    !wc -l data/train.jsonl data/val.jsonl data/test_held_out.jsonl
else:
    print('Preparing dataset (this takes ~10 minutes)...')
    !python data/prepare.py

## Step 6 — Inspect a few training examples

Critical sanity check before training. Verify:
1. The `<start_of_turn>user` / `<start_of_turn>model` tags are present
2. Tool schemas appear inside the user turn
3. The model turn contains valid JSON

If the format looks wrong here, fix it before spending 90 minutes training on bad data.

In [ ]:
!python data/inspect.py --n 2

## Step 7 — Run training

This is the main event. `train.py` will:
1. Load Gemma 4 E4B in 4-bit (uses ~7GB of the T4's 16GB)
2. Attach LoRA adapters (~20M trainable parameters)
3. Train for 1 epoch (~90 minutes)
4. Save checkpoints every 200 steps (so disconnects don't lose progress)
5. Push the final adapter to Hugging Face Hub

**Watch for:**
- `trainable params: ~20M || all params: ~4B || trainable%: ~0.5%` — confirms LoRA is set up
- Loss starting around 1.5–2.0 and dropping to 0.3–0.6 — confirms training is working
- Step numbers increasing — confirms it's not frozen

Track live metrics at: wandb.ai → your project → `tool-use-specialist`

In [ ]:
!python train.py --config configs/train.yaml

## Step 8 — Verify the adapter is on HuggingFace Hub

After training completes, this confirms the adapter was successfully pushed.

In [ ]:
from huggingface_hub import HfApi

api = HfApi(token=os.environ['HF_TOKEN'])
repo_id = 'sahilmdeshmukh/gemma-4-e4b-tool-use-lora'

try:
    files = api.list_repo_files(repo_id)
    print(f'Adapter repo exists: {repo_id}')
    print('Files:')
    for f in files:
        print(f'  {f}')
except Exception as e:
    print(f'Could not find repo: {e}')
    print('Training may not have completed or push failed.')

## Step 9 — Quick inference test

Load your fine-tuned adapter and run a sample tool call to confirm it works before moving to the eval harness (Day 3).

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

BASE_MODEL  = 'google/gemma-4-e4b-it'
ADAPTER     = 'sahilmdeshmukh/gemma-4-e4b-tool-use-lora'
HF_TOKEN    = os.environ['HF_TOKEN']

# Load base in 4-bit + adapter on top
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, token=HF_TOKEN)
base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb,
                                             device_map='auto', token=HF_TOKEN)
model = PeftModel.from_pretrained(base, ADAPTER, token=HF_TOKEN)
model.eval()

# Sample query
tools = [{
    'name': 'get_weather',
    'description': 'Get current weather for a location',
    'parameters': {
        'type': 'object',
        'properties': {
            'location': {'type': 'string'},
            'unit': {'type': 'string', 'enum': ['celsius', 'fahrenheit']}
        },
        'required': ['location']
    }
}]

import json
system_content = (
    'You have access to the following functions. Use them if required:\n\n'
    + json.dumps(tools, indent=2)
    + '\n\nRespond with a JSON object: {"name": "...", "arguments": {...}}'
)

messages = [
    {'role': 'user', 'content': f'{system_content}\n\nWhat is the weather in Mumbai in celsius?'}
]

inputs = tokenizer.apply_chat_template(messages, return_tensors='pt', add_generation_prompt=True)
inputs = inputs.to(model.device)

with torch.no_grad():
    outputs = model.generate(inputs, max_new_tokens=128, temperature=0.1, do_sample=True)

response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
print('Model output:')
print(response)
print()

# Try to parse as JSON
try:
    parsed = json.loads(response.strip())
    print('Valid JSON tool call!')
    print(f'  Function: {parsed["name"]}')
    print(f'  Args:     {parsed["arguments"]}')
except json.JSONDecodeError:
    print('Output is not valid JSON — may need more training or prompt tuning.')